Пролог: модели типа Image-Text-to_Text с Inference Providers

Осталось собрать данные по искомой теме, разметить их и сопоставить с результатом ИИ.

Ещё было бы неплохо выявить, какие ссылки не умеет смотреть чат, чтобы не взрываться на ошибках.

In [16]:
%pip install openai
%pip install pydantic_core

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
import os
from openai import OpenAI

In [18]:


model_1 = "Qwen/Qwen3-VL-8B-Instruct:novita"
model_2 = "google/gemma-3-27b-it:nebius"

In [ ]:
from typing_extensions import Literal
                                                      # простая реализация запроса
                                                      # можно посмотреть отличие работы ролей
def get_model_simple_response(model_name: str, prompt: str, role: Literal["user", "system"] = "user") -> str:
    client = OpenAI(
        base_url="https://router.huggingface.co/v1",  # базовый URL
        api_key="HF_KEY",             # API-ключ для huggyface
    )

    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": role,
                "content": prompt
            }
        ],
    )

    return completion.choices[0].message.content

In [20]:
                                                      # задать простой промпт
simple_prompt = "Вертолёты -- это воздушные аппараты? Отвечай да или нет"
#simple_prompt = "Почему модель ответила, что лимоны зелёные?"

In [21]:
print(get_model_simple_response(model_1, simple_prompt, "user"))

Да


In [22]:
# @title
# Такой промпт вызовет ошибку из-за отстутствия чередования ролей в сообщениях
# Подразумевается, что user и assistant (ответы модели) чередуются
# Собственно, модель негодует и дропает ошибку

wrong_prompt = [
    {
        "role": "system", "content": [
            {
                "type": "text",
                "text": "Отвечай только 'да' или 'нет'."
            }
        ]
    },
    {
        "role": "user", "content": [
            {
                "type": "text",
                "text": "Вертолёты -- это воздушные аппараты?"
            }
        ]
    },
    {
        "role": "user", "content": [
            {
                "type": "text",
                "text": "Лимоны зелёные?"
            },
            {
                "type": "image_url",
                "image_url": "https://cdn.metro-cc.ru/ru/ru_pim_536947001001_01.png"
            }
        ]
    },
    {
        "role": "user", "content": [
            {
                "type": "text",
                "text": "А апельсины?"
            }
        ]
    },
]

In [ ]:
from typing import List, Dict
                                                      # принимает детализированный запрос
def get_model_response(model_name: str, prompt: List[Dict[str, str]]) -> str:
    client = OpenAI(
        base_url="https://router.huggingface.co/v1",  # базовый URL
        api_key="HF_KEY",             # API-ключ для huggyface
    )

    completion = client.chat.completions.create(
        model=model_name,
        messages=prompt,
    )

    return completion.choices[0].message.content

In [24]:
                                                      # system для ограничений
def add_instruction_to_prompt(instruction):
    return {"role": "system", "content": instruction}
                                                      # user для вопросов
def add_questions_to_prompt(questions):
    return {"role": "user", "content": "\n".join(questions)}

def create_full_prompt(instruction, questions):
    system_message = add_instruction_to_prompt(instruction)
    user_message = add_questions_to_prompt(questions)
    return [system_message, user_message]

In [25]:
                                                      # задать ограничения
instruction = "Отвечай только 'да' или 'нет'."
                                                      # задать список вопросов
questions = [
    "Вертолёты -- это воздушные аппараты?",
    "Лимоны зелёные?",
    "А апельсины?"
]
                                                      # создать сложный промпт
prompt_with_many_questions = create_full_prompt(instruction, questions)
#print(prompt_with_many_questions)

In [26]:
print(get_model_simple_response(model_1, simple_prompt, "user"))

Да


In [27]:
print(get_model_response(model_2, prompt_with_many_questions))

да
да
да



In [28]:
# отправка ссылки на изображение через текст не приводит к анализу изображения
# модель анализирует сопутствующие данные (на картинке изображён лимон)

img_url = "https://cdn.metro-cc.ru/ru/ru_pim_536947001001_01.png"
img_prompt = f"Что изображено на картинке по ссылке {img_url}?"

print(get_model_simple_response(model_1, img_prompt));

На картинке, указанной по ссылке **https://cdn.metro-cc.ru/ru/ru_pim_536947001001_01.png**, изображена **упаковка продукта** — это **пачка чая «Метро» (Metro)**, а точнее — **чай черный с лимоном и мятой**.

### Описание:
- **Бренд**: Metro (Метро) — российская торговая марка, известная своими продуктами для дома и кухни.
- **Тип чая**: Черный чай с добавлением лимона и мяты.
- **Упаковка**: Пачка (обычно 25 или 50 шт. пакетиков), в зависимости от серии.
- **Цветовая гамма**: Белый фон с красными и зелёными элементами, характерными для бренда Metro.
- **Дизайн**: На упаковке присутствует название продукта, логотип Metro, изображение лимона и мяты, а также информация о составе и способе заваривания.

Это стандартная упаковка чая, продаваемого в супермаркетах сети **«Метро»** в России. Такие изображения используются для каталогов, сайтов и рекламы.

> ⚠️ *Примечание*: Ссылка указывает на CDN-сервер Metro, что подтверждает принадлежность к их продуктам. Изображение не является художествен

In [29]:
# из-за обязательного чередования ответов пользователя и ассистента
# (а так же формата ввода запроса)
# для обработки нескольких изображений можно использовать только один запрос
# (а не "что это на картинках 1 и 2?" и "синее ли солнце на рисунке?")

In [ ]:
                                                      # реализация запроса со множеством изображений
def get_model_images_response(model_name: str, message) -> str:
    client = OpenAI(
        base_url="https://router.huggingface.co/v1",  # базовый URL
        api_key="HF_KEY",             # API-ключ для huggyface
    )

    completion = client.chat.completions.create(
        model=model_name,
        messages=message
    )

    return completion.choices[0].message.content

In [36]:
                                                      # system для ограничений
def add_instruction_to_prompt_with_images(instruction):
    return {"role": "system", "content": [{"type": "text", "text": instruction}]}
                                                      # user для вопросов
def add_question_to_prompt_with_images(question, image_urls):
    user_content = [{"type": "text", "text": question}]
    for url in image_urls:
        user_content.append({"type": "image_url", "image_url": {"url": url}})
    return {"role": "user", "content": user_content}

def create_full_prompt_with_images(instruction, question, image_urls):
    system_message = add_instruction_to_prompt_with_images(instruction)
    user_message = add_question_to_prompt_with_images(question, image_urls)
    return [system_message, user_message]

In [37]:
                                                      # задать ограничения
instruction = "Отвечай только 'да' или 'нет' для каждой картинки."
                                                      # задать вопрос вопросов
question = "На картинке изображён лимон?"
                                                      # задать список изображений
img_urls = [
                                                      # действительно лимон
    "https://cdn.metro-cc.ru/ru/ru_pim_536947001001_01.png",
                                                      # вертолёт
    "https://avatars.mds.yandex.net/i?id=eb4422f165e38aa8254cfa2236767872_l-11008180-images-thumbs&n=13"
]

prompt_with_images = create_full_prompt_with_images(instruction, question, img_urls)
#print(prompt_with_images)

In [38]:
print(get_model_images_response(model_1, prompt_with_images))

да
нет


In [ ]:
print(get_model_images_response(model_2, prompt_with_images))

да
нет


In [ ]:
                                                      # пример с вертолётами
instruction = "Отвечай только 'гражданский' или 'военный' для каждой картинки."

question = "Какой вертолёт изображён на картинке?"

img_urls = [
    "https://avatars.mds.yandex.net/i?id=71813b6fb1c6b1e822b331f5436a8970_l-5341701-images-thumbs&n=13",
    "https://rostec.ru/upload/iblock/ed8/ed8b4cfe5f974f30ee52e05f60741212.jpg",
    "https://avatars.mds.yandex.net/i?id=ce2568bb6d1523d2ed0398e9a340690f_l-7455892-images-thumbs&n=13",
]

prompt_with_images = create_full_prompt_with_images(instruction, question, img_urls)
print(get_model_images_response(model_1, prompt_with_images))

гражданский
гражданский
военный


In [ ]:
img_urls = [
    "https://cdn.profile.ru/wp-content/uploads/2021/07/Kamov_Ka-62_at_the_MAKS-2013_01.jpeg",
    #"https://cdnn21.img.ria.ru/images/07e4/0c/15/1590161537_0:0:2442:1628_1920x1280_80_0_0_a178ba6252d4015fd1e33da8538bae7c.jpg",
    #"https://i.pinimg.com/originals/88/b3/eb/88b3ebac3491f1fefbd250700862b68f.jpg",
    "https://topwar.ru/uploads/posts/2016-11/1479297540_2.jpeg",
    "https://cdnstatic.rg.ru/resize800x533/uploads/images/gallery/b2fc7611/2_122e2fd6.jpg",
]

prompt_with_images = create_full_prompt_with_images(instruction, question, img_urls)
print(get_model_images_response(model_2, prompt_with_images))

гражданский
военный
гражданский


In [ ]:
from googleapiclient.discovery import build

def get_image_urls_list(page_count: int, search_query: str, img_per_page = 10):
    service = build("customsearch", "v1", developerKey=userdata.get('GC_api'))

    image_links = []

    for i in range(0, page_count):
        res = (service.cse().list(
                q=search_query,                       # запрос
                cx=userdata.get('CS_cx'),             # поисковая система
                searchType='image',                   # тип элементов
                num = img_per_page,                   # количество элементов в выдаче (максимум 10)
                start = i * img_per_page + 1          # определяет начало выдачи
            ).execute()
        )

        image_links.extend([item['link'] for item in res.get('items', [])])

    return image_links

In [ ]:
import pprint
import random

url_list = get_image_urls_list(1, "Red apple")
labled_list = [(url, 'apple') for url in url_list]

url_list = get_image_urls_list(1, "Pomegranate fruit")
labled_list.extend([(url, 'pomegranate') for url in url_list])

random.shuffle(labled_list)
pprint.pprint(labled_list)

[('https://i5.walmartimages.com/seo/Fresh-Red-Delicious-Apple-Each_7320e63a-de46-4a16-9b8c-526e15219a12_3.e557c1ad9973e1f76f512b34950243a3.jpeg',
  'apple'),
 ('https://media.istockphoto.com/id/185262648/photo/red-apple-with-leaf-isolated-on-white-background.jpg?s=612x612&w=0&k=20&c=gUTvQuVPUxUYX1CEj-N3lW5eRFLlkGrU_cwwwOWxOh8=',
  'apple'),
 ('https://static.vecteezy.com/system/resources/previews/005/170/799/non_2x/red-apple-with-apple-slices-and-leaves-vitamins-healthy-food-fruit-on-a-white-background-realistic-3d-illustration-vector.jpg',
  'apple'),
 ('https://images.squarespace-cdn.com/content/v1/6750ca6b3bccca5a0d530520/a69d45b7-dab7-4417-9cc9-9388b739d04c/RAFLogoTransparent.png',
  'apple'),
 ('https://www.sciencelearn.org.nz/_next/image?url=https%3A%2F%2Fwww.datocms-assets.com%2F117510%2F1722379364-hero_red_apple_on_blue_white_bg_01.jpg%3Fw%3D1840%26h%3D1495.1466213344665&w=1920&q=85',
  'apple'),
 ('https://media.post.rvohealth.io/wp-content/uploads/2022/02/pomegranate-seeds-fr

In [ ]:
instruction = "Отвечай только 'яблоко' или 'гранат' для каждой картинки."

question = "Что изображено на картинке?"

img_urls = [fruit[0] for fruit in labled_list]

prompt_with_images = create_full_prompt_with_images(instruction, question, img_urls[4:5])
print(get_model_images_response(model_1, prompt_with_images))

яблоко


Поскольку часть картинок являются проблемными, перед тем, как скормить чату их список, придётся каждую проверить на валидность. Поскольку если среди 10 картинок ошибку вызывает одна, отлетает весь набор. Перед разметкой данных на яблоки и гранаты запустим безопасный запрос.

Раз уж мы всё равно обращаемся к нейронке для проверки изображения, сразу попросим её распознать, если возможно. Если нет, обработаем ошибку.

In [ ]:
import threading
                                                      # не поддерживает signal, поэтому испольуем потоки
def get_response_with_timeout(function, args=(), kwargs=None, timeout=100):
    if kwargs is None:
        kwargs = {}
    result = {}

    def target():
        try:
            result['response'] = function(*args, **kwargs)
        except Exception as e:
            result['error'] = e

    thread = threading.Thread(target=target)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        return 'timeout'                              # скорее всего завис
    else:
        if 'error' in result:
            print(result['error'])
            return 'error'
        return result.get('response')

In [ ]:
def get_safe_model_images_response(model_name: str, message) -> str:
    def api_call():
        client = OpenAI(
            base_url="https://router.huggingface.co/v1",
            api_key=userdata.get('HF_TOKEN'),
        )
        completion = client.chat.completions.create(
            model=model_name,
            messages=message,
        )
        return completion.choices[0].message.content

    return get_response_with_timeout(api_call)

In [ ]:
# @title
import pprint
import random

instruction = "Отвечай только 'яблоко' или 'гранат' для каждой картинки."
question = "Что изображено на картинке?"

labled_list = []

apple_list = get_image_urls_list(1, "Red apple")

for i in range(0, len(apple_list)):
    prompt_with_image = create_full_prompt_with_images(instruction, question, apple_list[i:i+1])
    model_answer = get_safe_model_images_response(model_1, prompt_with_image)
    print(model_answer)
    #if model_answer != 'error':
    labled_list.extend((apple_list[i], model_answer))

#pprint.pprint(labled_list)

#pomegranate_list = get_image_urls_list(100, "Pomegranate fruit")
#labled_list.extend([(url, 'pomegranate') for url in url_list])

#random.shuffle(labled_list)

яблоко
яблоко
timeout
timeout
timeout
яблоко
timeout
яблоко
яблоко
яблоко


In [ ]:
import concurrent.futures

def process_sublist(sublist, model):
    local_results = []
    i = 0
    for url in sublist:
        prompt_with_image = create_full_prompt_with_images(instruction, question, [url])
        model_answer = get_safe_model_images_response(model, prompt_with_image)
        print(f'{model_answer} -- {i}')              # помогает не сойти с ума во время ожидания
        i = i + 1
        local_results.extend([url, model_answer])
    return local_results

In [ ]:
def process_fruit_list(fruit_list, model, chunk_size = 10, max_workers = 5):
                                                      # разделение списка на части
    sublists = [fruit_list[i:i + chunk_size] for i in range(0, len(fruit_list), chunk_size)]

    local_results = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_sublist, sublist, model) for sublist in sublists]

        for future in concurrent.futures.as_completed(futures):
            result = future.result()
            local_results.extend(result)

In [ ]:
import pprint
import random
                                                      # всё на свете и с потоками
instruction = "Отвечай только 'яблоко' или 'гранат' для каждой картинки."
question = "Что изображено на картинке?"

                                                      # количество страниц ограничено квотой !!! 100 в день
page_count = 10

apple_list = get_image_urls_list(page_count, "Red apple")
pomegranate_list = get_image_urls_list(page_count, "Pomegranate fruit")

SecretNotFoundError: Secret GC_api does not exist.

In [ ]:
pprint.pprint(apple_list)

['https://i5.walmartimages.com/seo/Fresh-Red-Delicious-Apple-Each_7320e63a-de46-4a16-9b8c-526e15219a12_3.e557c1ad9973e1f76f512b34950243a3.jpeg',
 'https://greenacrefarmersmarket.com/cdn/shop/products/reddeliscious_700x.png?v=1585923131',
 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Red_Apple.jpg/960px-Red_Apple.jpg',
 'https://ichef.bbci.co.uk/images/ic/480xn/p07v2wjn.jpg.webp',
 'https://media.istockphoto.com/id/185262648/photo/red-apple-with-leaf-isolated-on-white-background.jpg?s=612x612&w=0&k=20&c=gUTvQuVPUxUYX1CEj-N3lW5eRFLlkGrU_cwwwOWxOh8=',
 'https://gallery.yopriceville.com/downloadfullsize/send/22537',
 'https://images.squarespace-cdn.com/content/v1/6750ca6b3bccca5a0d530520/a69d45b7-dab7-4417-9cc9-9388b739d04c/RAFLogoTransparent.png',
 'https://www.sciencelearn.org.nz/_next/image?url=https%3A%2F%2Fwww.datocms-assets.com%2F117510%2F1722379364-hero_red_apple_on_blue_white_bg_01.jpg%3Fw%3D1840%26h%3D1495.1466213344665&w=1920&q=85',
 'https://static.vecteezy.com/sys

In [ ]:
pre_parsed_apple_list = [
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple1.jpeg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple2.jpeg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple3.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple4.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple5.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple6.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple7.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple8.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple9.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple10.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple11.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple12.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple13.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple14.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple15.jpg'
]

In [ ]:
pprint.pprint(pomegranate_list)

['https://www.columbiatribune.com/gcdn/authoring/2012/12/04/NCDT/ghows-MO-7a35dcf7-6193-4659-83d8-fbbeb06af9b4-37258d1a.jpeg?width=660&height=547&fit=crop&format=pjpg&auto=webp',
 'https://www.thegardener.co.za/wp-content/uploads/2024/12/14-1024x1024.png',
 'https://media.post.rvohealth.io/wp-content/uploads/2022/02/pomegranate-seeds-fruit-732x549-thumbnail.jpg',
 'https://post.healthline.com/wp-content/uploads/2022/02/pomegranate-seeds-fruit-1296x728-header.jpg',
 'https://media.post.rvohealth.io/wp-content/uploads/2022/02/pomegranate-seeds-fruit-732x549-thumbnail-732x549.jpg',
 'https://www.unlockfood.ca/EatRightOntario/media/Website-images-resized/I%E2%80%99ve-heard-that-pomegranates-have-many-health-benefits-resized.jpg',
 'https://cdn.britannica.com/96/201196-050-C0441508/Batch-pomegranate-fruits.jpg',
 'https://www.biovie.fr/modules/prestablog/views/img/grid-for-1-7/up-img/thumb_624.jpg?0020e2ca9a018d7f4d52febd25ec3cbd',
 'https://upload.wikimedia.org/wikipedia/commons/thumb/6/6a

In [ ]:
# @title
pre_parsed_pomegranate_list = [
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate1.jpeg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate2.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate3.png',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate4.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate5.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate6.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate7.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate8.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate9.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate10.png',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate11.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate12.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate13.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate14.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate15.jpg'
]

In [ ]:
                                                      # в модель 1
labled_apple_list_1 = process_fruit_list(pre_parsed_apple_list, model_1)

NameError: name 'process_fruit_list' is not defined

In [ ]:
labled_pomegranate_list_1 = process_fruit_list(pomegranate_list, model_1)

Error code: 402 - {'error': 'You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.'}
error -- 0
Error code: 402 - {'error': 'You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.'}
error -- 0
Error code: 402 - {'error': 'You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.'}
error -- 0
Error code: 402 - {'error': 'You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.'}
error -- 0
Error code: 402 - {'error': 'You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.'}
error -- 0
Error code: 402 - {'error': 'You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x 

KeyboardInterrupt: 

In [ ]:
pprint.pprint(labled_apple_list_1)

None


In [ ]:
pprint.pprint(labled_pomegranate_list_1)

None


In [ ]:
                                                      # в модель 2
labled_apple_list_2 = process_fruit_list(apple_list, model_2)
labled_pomegranate_list_2 = process_fruit_list(pomegranate_list, model_2)

In [ ]:
pprint.pprint(labled_apple_list_2)

In [ ]:
pprint.pprint(labled_pomegranate_list_2)

Для чистоты эсперимента можно объединить списки и перемешать ссылки, почистить ответы нейронки и снова загнать на распознавание, но это выглядит как оверкил.

In [ ]:
# Тут нужно проанализировать полученные ответы, сравнить их с маркерами и нарисовать кресты ошибок